<!-- MIGRATED_V2_TO_V3_NOTICE -->
> **ℹ️ This notebook is migrated from SageMaker Python SDK v2 to v3.**
>
> This is the v3-based version, and we recommend referring to and using this version. The SageMaker Python SDK v2 and v3 are **not backward compatible**, so v2 code will not run on a v3 installation.
>
> If you are looking for the original v2 version of this notebook, please go to the `v2-archive` branch and look for the notebook with the same name.


# Managed Spot Training for XGBoost (V3)


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

---

> **SageMaker Python SDK v3 note:** This notebook has been migrated to the SageMaker Python SDK **v3**. Managed Spot Training for XGBoost now uses `ModelTrainer` (from `sagemaker-train`) instead of the v2 `Estimator`/`sagemaker.xgboost.XGBoost`. Spot training is configured through the structured `Compute` (`enable_managed_spot_training=True`), `StoppingCondition` (`max_runtime_in_seconds` / `max_wait_time_in_seconds`), and `CheckpointConfig` (`s3_uri`) config objects. Automatic Model Tuning uses the v3 `HyperparameterTuner` (from `sagemaker.train.tuner`) with parameter ranges from `sagemaker.core.parameter`, and container/session helpers come from `sagemaker.core`. Install with `pip install sagemaker` (v3).

---


This notebook shows usage of SageMaker Managed Spot infrastructure for XGBoost training. Below we show how Spot instances can be used for the 'algorithm mode' and 'script mode' training methods with the XGBoost container. 

[Managed Spot Training](https://docs.aws.amazon.com/sagemaker/latest/dg/model-managed-spot-training.html) uses Amazon EC2 Spot instance to run training jobs instead of on-demand instances. You can specify which training jobs use spot instances and a stopping condition that specifies how long Amazon SageMaker waits for a job to run using Amazon EC2 Spot instances.

This notebook was tested in Amazon SageMaker Studio on a ml.t3.medium instance with Python 3 (Data Science) kernel.

In this notebook we will perform XGBoost training as described [here](). See the original notebook for more details on the data. 

### Setup variables and define functions

In [ ]:
!pip install sagemaker --upgrade --quiet

In [ ]:
%%time

import io
import os
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sagemaker_session = Session()
role = get_execution_role()
region = sagemaker_session.boto_region_name

# S3 bucket for saving code and model artifacts.
# Feel free to specify a different bucket here if you wish.
bucket = sagemaker_session.default_bucket()
prefix = "sagemaker/DEMO-xgboost-spot"
default_bucket_prefix = sagemaker_session.default_bucket_prefix

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

# customize to your bucket where you have would like to store the data

### Fetching the dataset

In [ ]:
%%time
s3 = boto3.client("s3")
# Load the dataset
FILE_DATA = "abalone"
s3.download_file(
    f"sagemaker-example-files-prod-{region}",
    f"datasets/tabular/uci_abalone/abalone.libsvm",
    FILE_DATA,
)
sagemaker_session.upload_data(FILE_DATA, bucket=bucket, key_prefix=prefix + "/train")
sagemaker_session.upload_data(FILE_DATA, bucket=bucket, key_prefix=prefix + "/validation")

### Obtaining the latest XGBoost container
We obtain the new container by specifying the framework version (1.7-1). This version specifies the upstream XGBoost framework version (1.7) and an additional SageMaker version (1). If you have an existing XGBoost workflow based on the previous (1.0-1, 1.2-2, 1.3-1 or 1.5-1) container, this would be the only change necessary to get the same workflow working with the new container.

In v3, the `image_uris` helper lives under `sagemaker.core`.

In [ ]:
from sagemaker.core import image_uris

container = image_uris.retrieve("xgboost", region, "1.7-1")

### Training the XGBoost model

After setting training parameters, we kick off training, and poll for status until training is completed, which in this example, takes few minutes.

To run training on SageMaker in v3 we construct a `ModelTrainer` (from `sagemaker-train`), which replaces the v2 `Estimator`. It accepts structured configuration objects:

* __training_image__: The XGBoost container image URI retrieved above.
* __role__: Role ARN.
* __hyperparameters__: A dictionary passed to the algorithm as hyperparameters.
* __compute__: A `Compute` object describing the instance type/count (and, below, the managed spot training flag). __Note__: This particular mode does not currently support training on GPU instance types.
* __sagemaker_session__ *(optional)*: The session used to train on SageMaker.

If Spot instances are used, the training job can be interrupted, causing it to take longer to start or finish. If a training job is interrupted, a checkpointed snapshot can be used to resume from a previously saved point and can save training time (and cost).

To enable checkpointing for Managed Spot Training using SageMaker XGBoost we configure three things through v3 config objects: 

1. Set `Compute(enable_managed_spot_training=True)` - the structured v3 equivalent of the v2 `use_spot_instances` boolean. 

2. Set `StoppingCondition(max_wait_time_in_seconds=...)` - the amount of time you are willing to wait for Spot infrastructure to become available. Some instance types are harder to get at Spot prices and you may have to wait longer. You are not charged for time spent waiting for Spot infrastructure to become available, you're only charged for actual compute time spent once Spot instances have been successfully procured. 

3. Set `CheckpointConfig(s3_uri=...)` - this tells SageMaker an S3 location where to save checkpoints. While not strictly necessary, checkpointing is highly recommended for Managed Spot Training jobs due to the fact that Spot instances can be interrupted with short notice and using checkpoints to resume from the last interruption ensures you don't lose any progress made before the interruption.

Feel free to toggle the `use_spot_instances` variable to see the effect of running the same job using regular (a.k.a. "On Demand") infrastructure.

Note that `max_wait_time_in_seconds` can be set if and only if managed spot training is enabled and must be greater than or equal to `max_runtime_in_seconds`.

In [ ]:
import time
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    Compute,
    StoppingCondition,
    CheckpointConfig,
    OutputDataConfig,
    InputData,
)

hyperparameters = {
    "max_depth": "5",
    "eta": "0.2",
    "gamma": "4",
    "min_child_weight": "6",
    "subsample": "0.7",
    "objective": "reg:squarederror",
    "num_round": "50",
    "verbosity": "2",
}

instance_type = "ml.m5.4xlarge"
output_path = "s3://{}/{}/{}/output".format(bucket, prefix, "abalone-xgb")
content_type = "libsvm"

job_name = "DEMO-xgboost-spot-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
print("Training job", job_name)

use_spot_instances = True
max_run = 3600
max_wait = 7200 if use_spot_instances else None
checkpoint_s3_uri = (
    "s3://{}/{}/checkpoints/{}".format(bucket, prefix, job_name) if use_spot_instances else None
)
print("Checkpoint path:", checkpoint_s3_uri)

# Managed spot training + stopping condition are expressed through structured v3 configs.
compute = Compute(
    instance_type=instance_type,
    instance_count=1,
    volume_size_in_gb=5,  # 5 GB
    enable_managed_spot_training=use_spot_instances,
)
stopping_condition = StoppingCondition(
    max_runtime_in_seconds=max_run,
    max_wait_time_in_seconds=max_wait,
)
checkpoint_config = (
    CheckpointConfig(s3_uri=checkpoint_s3_uri) if use_spot_instances else None
)

model_trainer = ModelTrainer(
    training_image=container,
    role=role,
    sagemaker_session=sagemaker_session,
    hyperparameters=hyperparameters,
    compute=compute,
    stopping_condition=stopping_condition,
    checkpoint_config=checkpoint_config,
    output_data_config=OutputDataConfig(s3_output_path=output_path),
    base_job_name=job_name,
)

train_input = InputData(
    channel_name="train",
    data_source="s3://{}/{}/{}".format(bucket, prefix, "train"),
    content_type="libsvm",
)
model_trainer.train(input_data_config=[train_input])

### Savings
Towards the end of the job you should see two lines of output printed:

- `Training seconds: X` : This is the actual compute-time your training job spent
- `Billable seconds: Y` : This is the time you will be billed for after Spot discounting is applied.

If you enabled managed spot training, then you should see a notable difference between `X` and `Y` signifying the cost savings you will get for having chosen Managed Spot Training. This should be reflected in an additional line:
- `Managed Spot Training savings: (1-Y/X)*100 %`

### Train with Automatic Model Tuning ([HPO](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html)) <a id='AMT'></a> and Spot Training enabled 
***
You could also train with Amazon SageMaker Automatic Model Tuning. AMT, also known as hyperparameter tuning, finds the best version of a model by running many training jobs on your dataset using the algorithm and ranges of hyperparameters that you specify. It then chooses the hyperparameter values that result in a model that performs the best, as measured by a metric that you choose. We will use the v3 `HyperparameterTuner` (from `sagemaker.train.tuner`) object to interact with Amazon SageMaker hyperparameter tuning APIs.
        
The code sample below shows you how to use the v3 `HyperparameterTuner` and a spot-enabled `ModelTrainer` together. Parameter ranges come from `sagemaker.core.parameter`.
***

In [ ]:
from sagemaker.core.parameter import ContinuousParameter, IntegerParameter
from sagemaker.core.common_utils import name_from_base
from sagemaker.train.tuner import HyperparameterTuner


# You can select from the hyperparameters supported by the model, and configure ranges of values to be searched for training the optimal model.(https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning-define-ranges.html)
hyperparameter_ranges = {
    "max_depth": IntegerParameter(0, 10, scaling_type="Auto"),
    "num_round": IntegerParameter(1, 4000, scaling_type="Auto"),
    "alpha": ContinuousParameter(0, 2, scaling_type="Auto"),
    "subsample": ContinuousParameter(0.5, 1, scaling_type="Auto"),
    "min_child_weight": ContinuousParameter(0, 120, scaling_type="Auto"),
    "gamma": ContinuousParameter(0, 5, scaling_type="Auto"),
    "eta": ContinuousParameter(0.1, 0.5, scaling_type="Auto"),
}

# Increase the total number of training jobs run by AMT, for increased accuracy (and training time).
max_jobs = 6
# Change parallel training jobs run by AMT to reduce total training time, constrained by your account limits.
# if max_jobs=max_parallel_jobs then Bayesian search turns to Random.
max_parallel_jobs = 2

hp_tuner = HyperparameterTuner(
    model_trainer,
    "validation:rmse",
    hyperparameter_ranges,
    max_jobs=max_jobs,
    max_parallel_jobs=max_parallel_jobs,
    objective_type="Minimize",
    base_tuning_job_name=job_name,
)

# Launch a SageMaker Tuning job to search for the best hyperparameters
# In this case, the tuner requires a `validation` channel to emit the validation:rmse metric.
# Since we only created a `train` channel, we re-use it for validation.
validation_input = InputData(
    channel_name="validation",
    data_source="s3://{}/{}/{}".format(bucket, prefix, "train"),
    content_type="libsvm",
)
hp_tuner.tune(inputs=[train_input, validation_input])

## Enabling checkpointing for script mode

An additional mode of operation is to run customizable scripts as part of the training and inference jobs. See [this notebook](./xgboost_abalone_dist_script_mode.ipynb) for details on how to setup script mode. 

Here we highlight the specific changes that would enable checkpointing and use Spot instances. 

Checkpointing in script mode for SageMaker XGBoost can be performed using the convenience helpers in the `sagemaker_xgboost_container.checkpointing` module: 

- `checkpointing.train`: a convenience wrapper around `xgb.train` that loads any existing checkpoint, resumes from the last completed iteration, and registers a callback that saves a checkpoint every round. 

- `checkpointing.load_checkpoint` / `checkpointing.save_checkpoint`: the lower-level building blocks used by `checkpointing.train`. 

These read/write the checkpoint directory, which in the accompanying `abalone.py` script is set to `/opt/ml/checkpoints`. 

The relevant part of `abalone.py` looks like the following:

---------
```
from sagemaker_xgboost_container import checkpointing

CHECKPOINTS_DIR = '/opt/ml/checkpoints'   # default location for Checkpoints
train_args = dict(
    params=train_hp,
    dtrain=dtrain,
    evals=watchlist,
    num_boost_round=args.num_round,
)
bst = checkpointing.train(train_args, checkpoint_dir=CHECKPOINTS_DIR)
```

### Using ModelTrainer for XGBoost script mode

In v3 the framework-specific `sagemaker.xgboost.XGBoost` estimator is replaced by the generic `ModelTrainer`. We point it at the same XGBoost container image (retrieved via `image_uris.retrieve`, training scope) and supply our script through a `SourceCode` object (`entry_script="abalone.py"`). We also pass the IAM role, the compute configuration, and the dictionary of hyperparameters that we want to pass to our script.

In [ ]:
from sagemaker.core.training.configs import SourceCode

job_name = "DEMO-xgboost-regression-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())
print("Training job", job_name)
checkpoint_s3_uri = (
    "s3://{}/{}/checkpoints/{}".format(bucket, prefix, job_name) if use_spot_instances else None
)
print("Checkpoint path:", checkpoint_s3_uri)

# The script-mode training image uses the same XGBoost framework version.
script_mode_image = image_uris.retrieve(
    "xgboost", region, "1.7-1", image_scope="training"
)

xgb_script_mode_trainer = ModelTrainer(
    training_image=script_mode_image,
    role=role,
    sagemaker_session=sagemaker_session,
    source_code=SourceCode(source_dir=".", entry_script="abalone.py"),
    hyperparameters=hyperparameters,
    compute=Compute(
        instance_type=instance_type,
        instance_count=1,
        volume_size_in_gb=5,
        enable_managed_spot_training=use_spot_instances,
    ),
    stopping_condition=StoppingCondition(
        max_runtime_in_seconds=max_run,
        max_wait_time_in_seconds=max_wait,
    ),
    checkpoint_config=(
        CheckpointConfig(s3_uri=checkpoint_s3_uri) if use_spot_instances else None
    ),
    output_data_config=OutputDataConfig(
        s3_output_path="s3://{}/{}/{}/output".format(bucket, prefix, "xgboost-script-mode")
    ),
    base_job_name=job_name,
)

Training is as simple as calling `train` on the `ModelTrainer`. This will start a SageMaker Training job that will download the data, invoke the entry point code (in the provided script file), and save any model artifacts that the script creates. In this case, the script requires a `train` and a `validation` channel. Since we only created a `train` channel, we re-use it for validation. 

In [ ]:
xgb_script_mode_trainer.train(input_data_config=[train_input, validation_input])

As previously stated, the `ModelTrainer` can also be passed to the v3 `HyperparameterTuner` object to interact with the Amazon SageMaker hyperparameter tuning APIs and create a Hyperparameter Tuning Job. Hyperparameters are automatically tuned which in most cases results in a more accurate model.

Unlike the built-in algorithm mode (whose container publishes `validation:rmse` automatically), a script-mode tuning job runs a generic training image, so we tell the tuner how to read the objective metric from the training logs via `metric_definitions`. The regex matches the `validation-rmse:<value>` lines emitted by the XGBoost checkpointing evaluation callback in `abalone.py`.

In [ ]:
hp_tuner = HyperparameterTuner(
    xgb_script_mode_trainer,
    "validation:rmse",
    hyperparameter_ranges,
    metric_definitions=[
        {"Name": "validation:rmse", "Regex": "validation-rmse:([0-9\\.]+)"}
    ],
    max_jobs=max_jobs,
    max_parallel_jobs=max_parallel_jobs,
    objective_type="Minimize",
    base_tuning_job_name=job_name,
)

# Launch a SageMaker Tuning job to search for the best hyperparameters
# In this case, the tuner requires a `validation` channel to emit the validation:rmse metric.
# Since we only created a `train` channel, we re-use it for validation.
hp_tuner.tune(inputs=[train_input, validation_input])

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/build_and_train_models|sm-managed_spot_training_xgboost|sm-managed_spot_training_xgboost.ipynb)
